In [ ]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / ".git").exists():
            return p
    raise RuntimeError("Could not locate project root (no .git found above cwd)")

PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"


# Synthetic Data Generation for Drift Testing

This notebook generates synthetic variants of `training_data_with_features.zarr` for testing the drift detection pipeline.

Implemented synthetic perturbation methods, each independently configurable and swept over a severity list:
- **Mean/variance shift** — mean shift and variance (std) scale are decoupled, so pure-mean, pure-variance, and combined drift can each be tested.
- **Covariance shift** — couples a subset of bands via a near-identity transform matrix.
- **Noise injection** — adds Gaussian noise scaled to per-band std.

Every method also supports:
- **Spatial subsetting** (`*_PIXEL_FRACTION`) — restrict the perturbation to a random subset of pixels (localized/regional drift) instead of the whole scene.
- **Class-conditional filtering** (`*_CLASS_FILTER`) — restrict the perturbation to pixels of a single disturbance class (0 or 1), which simulates *concept* drift (X|y changes, labels untouched) rather than plain covariate drift.

Two generation modes:
1. **Baseline replication mode** (intentional): replicate one baseline year across all years, then perturb selected years. Unperturbed years are exact duplicates of the baseline year by design — this isolates the perturbation's effect from real year-to-year variation.
2. **Per-year original mode**: keep each year's original data, then perturb selected years.

Defaults:
- Baseline year: 2018 (customizable)
- Perturbed year indices: [2, 5] (0-based, customizable)

This notebook only implements the dask-chunked pipeline (see "Scalable chunked pipeline" below). Earlier feature-matrix and eager raw-array approaches were removed — they materialized the full ~8.1M-pixel cube in memory and OOM'd at full scale.

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# -----------------------------
# User configuration
# -----------------------------
DATA_PATH = str(PROJECT_ROOT / "training_data_with_features.zarr")
OUTPUT_ROOT = PROJECT_ROOT / "synthetic_drift_data"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Default request-aligned settings
BASELINE_YEAR = 2018  # Customizable
PERTURB_YEAR_INDICES = [2, 5]  # 0-based year indices, customizable

DISTURBANCE_INVALID_VALUE = 255  # sentinel for "no label" in the disturbances array

# -----------------------------
# Mean/variance shift configuration
# mean_shift: fractional change to the band mean (0.10 = +10%), applied after centering.
# variance_scale: multiplicative change to the band std around the (shifted) mean (1.0 = no change).
# These are independent knobs: mean_shift=0 with variance_scale!=1 is a pure variance shift,
# and variance_scale=1 with mean_shift!=0 is a pure mean shift.
# -----------------------------
RED_BAND_INDEX = 3
MEAN_VARIANCE_SEVERITY_LEVELS = [
    {"mean_shift": 0.0, "variance_scale": 1.15},   # variance-only shift
    {"mean_shift": 0.10, "variance_scale": 1.0},   # mean-only shift
    {"mean_shift": 0.10, "variance_scale": 1.15},  # combined shift
]
MEAN_VARIANCE_PIXEL_FRACTION = 1.0  # 1.0 => all pixels; <1.0 => random subset (spatial/localized drift)
MEAN_VARIANCE_CLASS_FILTER = None   # None => all pixels; 0/1 => only that disturbance class (concept drift)

# Covariance shift configuration
COV_SHIFT_BAND_INDICES = [1, 2, ]  # subset to transform
COV_SHIFT_STRENGTH_LEVELS = [0.05, 0.15, 0.30]  # off-diagonal coupling strength, severity sweep
COV_SHIFT_PIXEL_FRACTION = 1.0
COV_SHIFT_CLASS_FILTER = 1

# Noise injection configuration
NOISE_SIGMA_RATIO_LEVELS = [0.04, 0.08, 0.16]  # sigma = ratio * per-band std, severity sweep
NOISE_PIXEL_FRACTION = 1.0
NOISE_CLASS_FILTER = None

RUN_CONFIG = {
    "random_state": RANDOM_STATE,
    "data_path": DATA_PATH,
    "baseline_year": BASELINE_YEAR,
    "perturb_year_indices": PERTURB_YEAR_INDICES,
    "mean_variance_shift": {
        "band_index": RED_BAND_INDEX,
        "severity_levels": MEAN_VARIANCE_SEVERITY_LEVELS,
        "pixel_fraction": MEAN_VARIANCE_PIXEL_FRACTION,
        "class_filter": MEAN_VARIANCE_CLASS_FILTER,
    },
    "covariance_shift": {
        "band_indices": COV_SHIFT_BAND_INDICES,
        "strength_levels": COV_SHIFT_STRENGTH_LEVELS,
        "pixel_fraction": COV_SHIFT_PIXEL_FRACTION,
        "class_filter": COV_SHIFT_CLASS_FILTER,
    },
    "noise_injection": {
        "sigma_ratio_levels": NOISE_SIGMA_RATIO_LEVELS,
        "pixel_fraction": NOISE_PIXEL_FRACTION,
        "class_filter": NOISE_CLASS_FILTER,
    },
}

print("Config ready.")
print(json.dumps(RUN_CONFIG, indent=2))

Config ready.
{
  "random_state": 42,
  "data_path": "training_data_with_features.zarr",
  "baseline_year": 2018,
  "perturb_year_indices": [
    2,
    5
  ],
  "mean_variance_shift": {
    "band_index": 3,
    "severity_levels": [
      {
        "mean_shift": 0.0,
        "variance_scale": 1.15
      },
      {
        "mean_shift": 0.1,
        "variance_scale": 1.0
      },
      {
        "mean_shift": 0.1,
        "variance_scale": 1.15
      }
    ],
    "pixel_fraction": 1.0,
    "class_filter": null
  },
  "covariance_shift": {
    "band_indices": [
      1,
      2,
      3,
      4
    ],
    "strength_levels": [
      0.05,
      0.15,
      0.3
    ],
    "pixel_fraction": 1.0,
    "class_filter": null
  },
  "noise_injection": {
    "sigma_ratio_levels": [
      0.04,
      0.08,
      0.16
    ],
    "pixel_fraction": 1.0,
    "class_filter": null
  }
}


In [4]:
# -----------------------------
# Load source dataset
# -----------------------------
load_start = time.time()
ds_source = xr.open_zarr(DATA_PATH)

if "s2_bands" not in ds_source:
    raise KeyError("Expected variable 's2_bands' in source dataset.")
if "year" not in ds_source.coords:
    raise KeyError("Expected coordinate 'year' in source dataset.")

s2_var = ds_source["s2_bands"]
if s2_var.ndim != 3:
    raise ValueError(f"Expected s2_bands with 3 dims (pixel, year, band). Got {s2_var.dims}.")

pixel_dim, year_dim, band_dim = s2_var.dims
year_values = ds_source[year_dim].values
n_pixels, n_years, n_bands = s2_var.shape

if BASELINE_YEAR not in set(int(v) for v in year_values):
    raise ValueError(f"Baseline year {BASELINE_YEAR} not found in dataset years: {year_values}.")

for idx in PERTURB_YEAR_INDICES:
    if idx < 0 or idx >= n_years:
        raise IndexError(f"Perturb year index {idx} is out of range [0, {n_years - 1}].")

baseline_idx = int(np.where(year_values == BASELINE_YEAR)[0][0])
perturb_year_values = [int(year_values[idx]) for idx in PERTURB_YEAR_INDICES]
synthetic_year_labels = [f"synthetic_{int(y)}" for y in year_values]

print(f"Dataset loaded in {time.time() - load_start:.1f}s")
print(f"s2_bands dims: {s2_var.dims}, shape: {s2_var.shape}")
print(f"Year values: {list(map(int, year_values))}")
print(f"Baseline year: {BASELINE_YEAR} (index={baseline_idx})")
print(f"Perturb indices: {PERTURB_YEAR_INDICES} -> years: {perturb_year_values}")

Dataset loaded in 0.8s
s2_bands dims: ('pixel', 'year', 's2_band'), shape: (8155205, 7, 7)
Year values: [2016, 2017, 2018, 2019, 2020, 2021, 2022]
Baseline year: 2018 (index=2)
Perturb indices: [2, 5] -> years: [2018, 2021]


In [5]:
# -----------------------------
# Perturbation utilities shared by the chunked pipeline
# -----------------------------
# Loaded once, eagerly: disturbances is tiny relative to s2_bands (uint8, pixel x year, ~57MB)
# and is used to build class-conditional (concept-drift) pixel masks per chunk below.
disturbances_full = ds_source["disturbances"].values


def _build_cov_transform_matrix(size, strength):
    # Near-identity matrix with controlled off-diagonal coupling.
    mat = np.eye(size, dtype=np.float64)
    if size > 1 and strength != 0:
        off_diag = strength / (size - 1)
        mat += off_diag * (np.ones((size, size), dtype=np.float64) - np.eye(size, dtype=np.float64))
    return mat


def years_suffix(years):
    return "-".join(str(y) for y in years)


def _class_filter_mask(pixel_start, pixel_stop, year_idx, class_filter):
    """Boolean mask over [pixel_start, pixel_stop) for a target year and class filter.

    class_filter=None selects every pixel (plain covariate drift). class_filter in (0, 1)
    selects only pixels whose disturbance label equals that class for this year (concept
    drift: X|y changes, y itself is left untouched).
    """
    if class_filter is None:
        return np.ones(pixel_stop - pixel_start, dtype=bool)
    labels = disturbances_full[pixel_start:pixel_stop, year_idx]
    return labels == class_filter


def _subset_row_indices(class_mask, pixel_fraction, rng_local):
    """Combine a class-conditional mask with a random pixel-fraction subsample within it."""
    eligible_idx = np.nonzero(class_mask)[0]
    if pixel_fraction >= 1.0 or len(eligible_idx) == 0:
        return eligible_idx
    n_sel = max(1, int(round(len(eligible_idx) * pixel_fraction)))
    return rng_local.choice(eligible_idx, size=n_sel, replace=False)


def validate_synthetic_dataset(ref_da, new_da):
    """Lazy, chunk-safe validation: only scalar reductions are computed, the full cube is
    never materialized.

    Compare against the mode's own pre-perturbation base (not the raw multi-year source):
    baseline-replication mode legitimately changes the missingness pattern relative to the
    source by design (each year takes on the baseline year's NaN pattern), so that
    comparison would falsely trip on every run. Comparing against the immediate
    pre-perturbation base isolates whether the *perturbation step itself* introduced new
    non-finite positions.
    """
    if new_da.shape != ref_da.shape:
        raise AssertionError("s2_bands shape changed unexpectedly")

    ref_finite = xr.apply_ufunc(np.isfinite, ref_da, dask="parallelized", output_dtypes=[bool])
    new_finite = xr.apply_ufunc(np.isfinite, new_da, dask="parallelized", output_dtypes=[bool])

    introduced_non_finite = bool((ref_finite & (~new_finite)).any().compute())
    if introduced_non_finite:
        raise AssertionError("Synthetic s2_bands introduced new non-finite values")


def changed_year_mask(ref_da, new_da):
    """Lazy, chunk-safe: True per year if any pixel/band actually changed relative to ref_da."""
    ref_finite = xr.apply_ufunc(np.isfinite, ref_da, dask="parallelized", output_dtypes=[bool])
    new_finite = xr.apply_ufunc(np.isfinite, new_da, dask="parallelized", output_dtypes=[bool])
    finite_pattern_changed = (ref_finite != new_finite).any(dim=[pixel_dim, band_dim])

    ref_filled = xr.apply_ufunc(np.nan_to_num, ref_da, dask="parallelized", output_dtypes=[ref_da.dtype])
    new_filled = xr.apply_ufunc(np.nan_to_num, new_da, dask="parallelized", output_dtypes=[new_da.dtype])
    values_changed = (np.abs(new_filled - ref_filled) > 0.0).any(dim=[pixel_dim, band_dim])

    changed = (finite_pattern_changed | values_changed).compute()
    return changed.values

## Scalable chunked pipeline

This is the only generation pipeline in this notebook. It processes `s2_bands` chunk-by-chunk
via dask and never materializes the full cube in memory.

For each of the 3 methods, this section sweeps over that method's `*_SEVERITY_LEVELS` list
(configured above) and generates one artifact per level, per mode — enough to build a
detection-power-vs-magnitude curve downstream. `*_PIXEL_FRACTION` and `*_CLASS_FILTER` are
applied consistently across every severity level for a given method.

Every generated artifact is validated (`validate_synthetic_dataset`) and diffed against its
own pre-perturbation base (`changed_year_mask`) before being logged to the manifest — both
checks are lazy dask reductions, so they never materialize the full cube either.

In [6]:
import shutil
import dask.array as da

# Chunking strategy: keep year and band whole, chunk by pixel.
PIXEL_CHUNK = 250_000

s2_lazy = ds_source["s2_bands"].chunk({pixel_dim: PIXEL_CHUNK, year_dim: -1, band_dim: -1})


def _year_mask_values(ny, target_indices):
    mask = np.zeros(ny, dtype=bool)
    mask[target_indices] = True
    return mask


def _mean_variance_shift_block(
    block, year_mask, band_index, mean_shift, variance_scale,
    pixel_fraction, class_filter, random_state, block_info=None,
):
    out = block.copy()
    pixel_start = block_info[None]["array-location"][0][0]
    chunk_loc = block_info[None]["chunk-location"][0]
    rng_local = np.random.default_rng(random_state + chunk_loc)
    n_pix = out.shape[0]

    target_years = np.where(year_mask)[0]
    for yi in target_years:
        class_mask = _class_filter_mask(pixel_start, pixel_start + n_pix, yi, class_filter)
        idx = _subset_row_indices(class_mask, pixel_fraction, rng_local)
        if len(idx) == 0:
            continue

        col = out[idx, yi, band_index]
        finite = np.isfinite(col)
        if not np.any(finite):
            continue

        mean_val = np.nanmean(col[finite])
        new_mean = mean_val * (1.0 + mean_shift)
        shifted = col.copy()
        shifted[finite] = new_mean + (col[finite] - mean_val) * variance_scale
        out[idx, yi, band_index] = shifted
    return out


def _cov_shift_block(
    block, year_mask, band_indices, strength,
    pixel_fraction, class_filter, random_state, block_info=None,
):
    out = block.copy()
    band_indices = np.array(band_indices, dtype=int)
    transform = _build_cov_transform_matrix(len(band_indices), strength)

    pixel_start = block_info[None]["array-location"][0][0]
    chunk_loc = block_info[None]["chunk-location"][0]
    rng_local = np.random.default_rng(random_state + chunk_loc)
    n_pix = out.shape[0]

    target_years = np.where(year_mask)[0]
    for yi in target_years:
        class_mask = _class_filter_mask(pixel_start, pixel_start + n_pix, yi, class_filter)
        idx = _subset_row_indices(class_mask, pixel_fraction, rng_local)
        if len(idx) == 0:
            continue

        subset = out[idx, yi, :][:, band_indices]
        row_ok = np.all(np.isfinite(subset), axis=1)
        if not np.any(row_ok):
            continue

        new_subset = subset.copy()
        new_subset[row_ok] = subset[row_ok] @ transform
        # idx is an integer array, so chained indexing (out[idx][:, yi, band_indices] = ...)
        # would write into a throwaway copy and silently do nothing. np.ix_ builds a single
        # combined index so the assignment writes into `out` directly.
        out[np.ix_(idx, [yi], band_indices)] = new_subset[:, None, :]
    return out


def _noise_block(
    block, year_mask, sigma_ratio,
    pixel_fraction, class_filter, random_state, block_info=None,
):
    out = block.copy()
    pixel_start = block_info[None]["array-location"][0][0]
    chunk_loc = block_info[None]["chunk-location"][0]
    rng_local = np.random.default_rng(random_state + chunk_loc)
    n_pix = out.shape[0]

    target_years = np.where(year_mask)[0]
    for yi in target_years:
        class_mask = _class_filter_mask(pixel_start, pixel_start + n_pix, yi, class_filter)
        idx = _subset_row_indices(class_mask, pixel_fraction, rng_local)
        if len(idx) == 0:
            continue

        block_year = out[idx, yi, :]
        band_std = np.nanstd(block_year, axis=0)
        band_std = np.where(np.isfinite(band_std), band_std, 0.0)
        sigma = sigma_ratio * np.maximum(band_std, 1e-8)
        noise = rng_local.normal(0.0, sigma, size=block_year.shape)
        out[idx, yi, :] = block_year + noise
    return out


def _apply_method_lazy(base_da, method_name, severity, pixel_fraction, class_filter):
    arr = base_da.data
    year_mask = _year_mask_values(n_years, PERTURB_YEAR_INDICES)

    if method_name == "mean_variance_shift":
        out = da.map_blocks(
            _mean_variance_shift_block,
            arr,
            year_mask=year_mask,
            band_index=RED_BAND_INDEX,
            mean_shift=severity["mean_shift"],
            variance_scale=severity["variance_scale"],
            pixel_fraction=pixel_fraction,
            class_filter=class_filter,
            random_state=RANDOM_STATE,
            dtype=arr.dtype,
        )
    elif method_name == "covariance_shift":
        out = da.map_blocks(
            _cov_shift_block,
            arr,
            year_mask=year_mask,
            band_indices=np.array(COV_SHIFT_BAND_INDICES, dtype=int),
            strength=severity,
            pixel_fraction=pixel_fraction,
            class_filter=class_filter,
            random_state=RANDOM_STATE,
            dtype=arr.dtype,
        )
    elif method_name == "noise_injection":
        out = da.map_blocks(
            _noise_block,
            arr,
            year_mask=year_mask,
            sigma_ratio=severity,
            pixel_fraction=pixel_fraction,
            class_filter=class_filter,
            random_state=RANDOM_STATE,
            dtype=arr.dtype,
        )
    else:
        raise ValueError(f"Unknown method: {method_name}")

    return xr.DataArray(
        out,
        dims=base_da.dims,
        coords=base_da.coords,
        attrs=base_da.attrs,
        name=base_da.name,
    )


def _save_ds_with_metadata(ds_synth, output_dir, artifact_stem, metadata):
    output_dir.mkdir(parents=True, exist_ok=True)
    zarr_path = output_dir / f"{artifact_stem}.zarr"
    json_path = output_dir / f"{artifact_stem}.metadata.json"

    if zarr_path.exists():
        shutil.rmtree(zarr_path)

    ds_synth.to_zarr(zarr_path, mode="w")

    with open(json_path, "w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2)

    return zarr_path, json_path

In [7]:
# -----------------------------
# Run scalable generation (full dataset)
# -----------------------------
METHODS = ["mean_variance_shift", "covariance_shift", "noise_injection"]

METHOD_SEVERITY_LEVELS = {
    "mean_variance_shift": MEAN_VARIANCE_SEVERITY_LEVELS,
    "covariance_shift": COV_SHIFT_STRENGTH_LEVELS,
    "noise_injection": NOISE_SIGMA_RATIO_LEVELS,
}
METHOD_PIXEL_FRACTION = {
    "mean_variance_shift": MEAN_VARIANCE_PIXEL_FRACTION,
    "covariance_shift": COV_SHIFT_PIXEL_FRACTION,
    "noise_injection": NOISE_PIXEL_FRACTION,
}
METHOD_CLASS_FILTER = {
    "mean_variance_shift": MEAN_VARIANCE_CLASS_FILTER,
    "covariance_shift": COV_SHIFT_CLASS_FILTER,
    "noise_injection": NOISE_CLASS_FILTER,
}


def _severity_stem(severity):
    if isinstance(severity, dict):
        return "-".join(f"{k}{v}" for k, v in severity.items())
    return str(severity)


def generate_mode_artifacts(mode_name, base_da, output_dir, baseline_year_for_meta=None):
    output_dir.mkdir(parents=True, exist_ok=True)
    rows = []

    for method_name in METHODS:
        pixel_fraction = METHOD_PIXEL_FRACTION[method_name]
        class_filter = METHOD_CLASS_FILTER[method_name]

        for severity in METHOD_SEVERITY_LEVELS[method_name]:
            da_method = _apply_method_lazy(base_da, method_name, severity, pixel_fraction, class_filter)

            ds_synth = ds_source.copy(deep=False)
            ds_synth["s2_bands"] = da_method
            ds_synth = ds_synth.assign_coords(
                synthetic_year_label=(year_dim, np.array(synthetic_year_labels, dtype=object))
            )

            stem = (
                f"synthetic_mode-{mode_name}"
                f"_method-{method_name}"
                f"_severity-{_severity_stem(severity)}"
                + (f"_class-{class_filter}" if class_filter is not None else "")
                + (f"_baseline-{baseline_year_for_meta}" if baseline_year_for_meta is not None else "")
                + f"_perturbed-{years_suffix(perturb_year_values)}"
            )

            meta = {
                "generated_at_utc": datetime.now(timezone.utc).isoformat(),
                "source_dataset": DATA_PATH,
                "mode": mode_name,
                "method": method_name,
                "severity": severity,
                "pixel_fraction": pixel_fraction,
                "class_filter": class_filter,
                "baseline_year": baseline_year_for_meta,
                "perturb_year_indices": PERTURB_YEAR_INDICES,
                "perturb_year_values": perturb_year_values,
                "synthetic_year_labels": synthetic_year_labels,
                "config": RUN_CONFIG,
                "engine": "xarray+dask_chunked",
            }

            zarr_path, json_path = _save_ds_with_metadata(ds_synth, output_dir, stem, meta)

            # Re-open what was actually persisted and validate/diff it against the mode's
            # own pre-perturbation base (base_da), lazily.
            ds_check = xr.open_zarr(zarr_path)
            validate_synthetic_dataset(base_da, ds_check["s2_bands"])
            changed_bool = changed_year_mask(base_da, ds_check["s2_bands"])
            changed_years = [int(year_values[i]) for i, flag in enumerate(changed_bool) if flag]

            rows.append(
                {
                    "mode": mode_name,
                    "method": method_name,
                    "severity": severity,
                    "pixel_fraction": pixel_fraction,
                    "class_filter": class_filter,
                    "baseline_year": baseline_year_for_meta,
                    "perturb_year_indices": PERTURB_YEAR_INDICES,
                    "perturb_year_values": perturb_year_values,
                    "changed_years": changed_years,
                    "zarr_path": str(zarr_path),
                    "metadata_path": str(json_path),
                }
            )
            print(f"Saved: {zarr_path}  (changed years: {changed_years})")

    return rows


artifact_rows = []

# Section A: replicate baseline year to all years lazily, then perturb selected years.
# Kept intentionally as an exact-duplicate baseline so "no drift" years carry zero
# natural year-to-year variation, isolating the perturbation's effect.
baseline_2d = s2_lazy.isel({year_dim: baseline_idx})
base_a = baseline_2d.expand_dims({year_dim: ds_source[year_dim]}).transpose(pixel_dim, year_dim, band_dim)
artifact_rows += generate_mode_artifacts(
    "baseline_replication",
    base_a,
    OUTPUT_ROOT / "section_a_baseline_replication",
    baseline_year_for_meta=BASELINE_YEAR,
)

# Section B: keep each year's real data, then perturb selected years.
artifact_rows += generate_mode_artifacts(
    "per_year_original",
    s2_lazy,
    OUTPUT_ROOT / "section_b_per_year_original",
)

manifest_df = pd.DataFrame(artifact_rows).sort_values(["mode", "method"]).reset_index(drop=True)
manifest_csv = OUTPUT_ROOT / "synthetic_generation_manifest.csv"
manifest_json = OUTPUT_ROOT / "synthetic_generation_manifest.json"
manifest_df.to_csv(manifest_csv, index=False)

with open(manifest_json, "w", encoding="utf-8") as handle:
    json.dump(
        {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "source_dataset": DATA_PATH,
            "baseline_year": BASELINE_YEAR,
            "perturb_year_indices": PERTURB_YEAR_INDICES,
            "perturb_year_values": perturb_year_values,
            "rows": manifest_df.to_dict(orient="records"),
        },
        handle,
        indent=2,
    )

print(f"Manifest CSV: {manifest_csv}")
print(f"Manifest JSON: {manifest_json}")
display(manifest_df)

C:\Users\bartu\AppData\Local\Temp\ipykernel_8732\520836388.py:166: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds_synth.to_zarr(zarr_path, mode="w")
c:\Users\bartu\Desktop\Fonda-scikit - Git\venv\Lib\site-packages\zarr\api\asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Saved: synthetic_drift_data\section_a_baseline_replication\synthetic_mode-baseline_replication_method-mean_variance_shift_severity-mean_shift0.0-variance_scale1.15_baseline-2018_perturbed-2018-2021.zarr  (changed years: [2018, 2021])


C:\Users\bartu\AppData\Local\Temp\ipykernel_8732\520836388.py:166: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds_synth.to_zarr(zarr_path, mode="w")
c:\Users\bartu\Desktop\Fonda-scikit - Git\venv\Lib\site-packages\zarr\api\asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Saved: synthetic_drift_data\section_a_baseline_replication\synthetic_mode-baseline_replication_method-mean_variance_shift_severity-mean_shift0.1-variance_scale1.0_baseline-2018_perturbed-2018-2021.zarr  (changed years: [2018, 2021])


C:\Users\bartu\AppData\Local\Temp\ipykernel_8732\520836388.py:166: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds_synth.to_zarr(zarr_path, mode="w")
c:\Users\bartu\Desktop\Fonda-scikit - Git\venv\Lib\site-packages\zarr\api\asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


...

C:\Users\bartu\AppData\Local\Temp\ipykernel_8732\520836388.py:166: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds_synth.to_zarr(zarr_path, mode="w")
c:\Users\bartu\Desktop\Fonda-scikit - Git\venv\Lib\site-packages\zarr\api\asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Saved: synthetic_drift_data\section_b_per_year_original\synthetic_mode-per_year_original_method-noise_injection_severity-0.08_perturbed-2018-2021.zarr  (changed years: [2018, 2021])


C:\Users\bartu\AppData\Local\Temp\ipykernel_8732\520836388.py:166: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds_synth.to_zarr(zarr_path, mode="w")
c:\Users\bartu\Desktop\Fonda-scikit - Git\venv\Lib\site-packages\zarr\api\asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Saved: synthetic_drift_data\section_b_per_year_original\synthetic_mode-per_year_original_method-noise_injection_severity-0.16_perturbed-2018-2021.zarr  (changed years: [2018, 2021])
Manifest CSV: synthetic_drift_data\synthetic_generation_manifest.csv
Manifest JSON: synthetic_drift_data\synthetic_generation_manifest.json


,mode,method,severity,pixel_fraction,class_filter,baseline_year,perturb_year_indices,perturb_year_values,changed_years,zarr_path,metadata_path
0,baseline_replication,covariance_shift,0.05,1.0,None,2018.0,"[2, 5]","[2018, 2021]","[2018, 2021]",synthetic_drift_data\section_a_baseline_replic...,synthetic_drift_data\section_a_baseline_replic...
1,baseline_replication,covariance_shift,0.15,1.0,None,2018.0,"[2, 5]","[2018, 2021]","[2018, 2021]",synthetic_drift_data\section_a_baseline_replic...,synthetic_drift_data\section_a_baseline_replic...
2,baseline_replication,covariance_shift,0.3,1.0,None,2018.0,"[2, 5]","[2018, 2021]","[2018, 2021]",synthetic_drift_data\section_a_baseline_replic...,synthetic_drift_data\section_a_baseline_replic...
3,baseline_replication,mean_variance_shift,"{'mean_shift': 0.0, 'variance_scale': 1.15}",1.0,None,2018.0,"[2, 5]","[2018, 2021]","[2018, 2021]",synthetic_drift_data\section_a_baseline_replic...,synthetic_drift_data\section_a_baseline_replic...
4,baseline_replication,mean_variance_shift,"{'mean_shift': 0.1, 'variance_scale': 1.0}",1.0,None,2018.0,"[2, 5]","[2018, 2021]","[2018, 2021]",synthetic_drift_data\section_a_baseline_replic...,synthetic_drift_data\section_a_baseline_replic...
5,baseline_replication,mean_variance_shift,"{'mean_shift': 0.1, 'variance_scale': 1.15}",1.0,None,2018.0,"[2, 5]","[2018, 2021]","[2018, 2021]",synthetic_drift_data\section_a_baseline_replic...,synthetic_drift_data\section_a_baseline_replic...
6,baseline_replication,noise_injection,0.04,1.0,None,2018.0,"[2, 5]","[2018, 2021]","[2018, 2021]",synthetic_drift_data\section_a_baseline_replic...,synthetic_drift_data\section_a_baseline_replic...
7,baseline_replication,noise_injection,0.08,1.0,None,2018.0,"[2, 5]","[2018, 2021]","[2018, 2021]",synthetic_drift_data\section_a_baseline_replic...,synthetic_drift_data\section_a_baseline_replic...
8,baseline_replication,noise_injection,0.16,1.0,None,2018.0,"[2, 5]","[2018, 2021]","[2018, 2021]",synthetic_drift_data\section_a_baseline_replic...,synthetic_drift_data\section_a_baseline_replic...
9,per_year_original,covariance_shift,0.05,1.0,None,NaN,"[2, 5]","[2018, 2021]","[2018, 2021]",synthetic_drift_data\section_b_per_year_origin...,synthetic_drift_data\section_b_per_year_origin...
